# HyperKvasir 23-class Qwen LoRA

Use Python (vlm). Run setup, model definitions and training configuration, then launch training. Both frozen CSV files are checked by SHA-256. For an ablation, give run_dir a new name and keep both CSVs fixed. Checkpoints resume after the last completed epoch. The comparison uses the whole frozen test partition, which includes previously evaluated images. Rare-class flags are reporting metadata. The test partition stays out of checkpoint selection.

In [ ]:
import csv
from collections import Counter
from hashlib import sha256
from io import BytesIO
from pathlib import Path, PurePosixPath
from random import Random
from zipfile import ZipFile

root = Path("E:/VLM")
data_dir = root / "data/hyperkvasir_23_v1"
manifest_path = data_dir / "split_manifest.csv"
rare_path = data_dir / "rare_labels.csv"
expected_hashes = ('11f36db99e3762145a3974c5fb57772b0120adf48cb353ee1e486cce9920c633', '609fc9ddfacd0db2584e6410158eaa59a7396c85036f4005d5bc82e6475d1688')
for path, expected in zip((manifest_path, rare_path), expected_hashes, strict=True):
    if sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f"Frozen artifact changed: {path}")
with rare_path.open(newline="", encoding="utf-8") as handle:
    rare_rows = list(csv.DictReader(handle))
labels = tuple(row["label"] for row in rare_rows)
assert len(set(labels)) == 23

splits = {name: [] for name in ("train", "validation", "test")}
fresh_test = []
with manifest_path.open(newline="", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        if row["status"] != "included":
            continue
        sample = (PurePosixPath(row["archive_member"]), row["label"])
        splits[row["split"]].append(sample)
        if row["split"] == "test" and not row["prior_use"]:
            fresh_test.append(sample)
for split, rows in splits.items():
    counts = Counter(label for _, label in rows)
    assert all(counts[row["label"]] == int(row[split]) for row in rare_rows)
    print(split, len(rows))
print("Previously unused test images:", len(fresh_test))
archive_path = root / "data/hyper-kvasir-labeled-images.zip"
if "image_archive" in globals():
    image_archive.close()
image_archive = ZipFile(archive_path)


In [ ]:
import gc
import re
from datetime import datetime
from statistics import median
from time import perf_counter

import torch
from PIL import Image, ImageOps
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_dir = root / "models/Qwen2.5-VL-3B-Instruct"
device = torch.device("cuda:0")
max_pixels = 448 * 448
prompt = "Archive this gastrointestinal endoscopy image. Reply with one exact label from: " + ", ".join(labels) + "."

def encode(processor, path, target=None):
    with Image.open(BytesIO(image_archive.read(path.as_posix()))) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    template_args = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prefix = processor.apply_chat_template(messages, add_generation_prompt=True, **template_args)
    if target is None:
        return prefix.to(device)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    inputs = processor.apply_chat_template(messages, add_generation_prompt=False, **template_args)
    prefix_length = prefix.input_ids.shape[1]
    if not torch.equal(inputs.input_ids[:, :prefix_length], prefix.input_ids):
        raise ValueError("Chat template changed the prompt tokens when the answer was appended.")
    targets = inputs.input_ids.clone()
    targets[:, :prefix_length] = -100
    if targets.shape[1] <= prefix_length:
        raise ValueError("No supervised answer tokens.")
    if "attention_mask" in inputs:
        targets[inputs.attention_mask == 0] = -100
    inputs["labels"] = targets
    return inputs.to(device)

@torch.inference_mode()
def evaluate(model, processor, rows):
    model.eval()
    pairs = []
    for path, truth in rows:
        inputs = encode(processor, path)
        tokens = model.generate(**inputs, max_new_tokens=16, do_sample=False)
        answer = processor.batch_decode(tokens[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
        answer = answer.strip().lower().removesuffix(".")
        label_text = answer.split("(", 1)[0].strip() if answer.endswith(")") else answer
        prediction = label_text if label_text in labels else "INVALID"
        pairs.append((truth, prediction))
    correct = sum(truth == pred for truth, pred in pairs)
    return correct, dict(Counter(pairs))



In [ ]:
from math import ceil, cos, pi
from peft import LoraConfig, PeftModel, get_peft_model

epochs = 6
accumulation = 4
run_dir = root / "runs/qwen_archive_23_lora_v1"
run_dir.mkdir(parents=True, exist_ok=True)
manifest_digest = sha256(manifest_path.read_bytes()).hexdigest()

def load_archive_model():
    processor = AutoProcessor.from_pretrained(model_dir, local_files_only=True)
    processor.image_processor.size = {"shortest_edge": 56 * 56, "longest_edge": max_pixels}
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_dir, dtype=torch.bfloat16, local_files_only=True
    ).to(device)
    return base, processor

def train_archive():
    checkpoints = sorted(path.parent for path in run_dir.glob("epoch_*/training.pt"))
    resume = checkpoints[-1] if checkpoints else None
    torch.manual_seed(2026)
    base, processor = load_archive_model()
    if resume:
        model = PeftModel.from_pretrained(base, resume, is_trainable=True)
    else:
        targets = [
            name for name, layer in base.named_modules()
            if ".language_model." in name and isinstance(layer, torch.nn.Linear)
            and name.rsplit(".", 1)[-1] in {"q_proj", "v_proj"}
        ]
        model = get_peft_model(base, LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.05, target_modules=targets, task_type="CAUSAL_LM"
        ))
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=1e-4)
    train_rows = splits["train"]
    class_counts = Counter(label for _, label in train_rows)
    class_weights = {label: (len(train_rows) / (len(labels) * count)) ** 0.5
                     for label, count in class_counts.items()}
    steps_per_epoch = ceil(len(train_rows) / accumulation)
    total_steps = epochs * steps_per_epoch
    warmup = max(1, round(total_steps * 0.05))
    config = (manifest_digest, expected_hashes[1], epochs, accumulation, prompt)
    start_epoch, step, best_score, best_epoch = 1, 0, -1.0, None
    if resume:
        state = torch.load(resume / "training.pt", map_location="cpu", weights_only=True)
        if state["config"] != config:
            raise ValueError("Training configuration differs from the saved run.")
        optimizer.load_state_dict(state["optimizer"])
        start_epoch, step = state["epoch"] + 1, state["step"]
        best_score, best_epoch = state["best_score"], state["best_epoch"]
        torch.set_rng_state(state["cpu_rng"])
        torch.cuda.set_rng_state(state["cuda_rng"], device)
    print(f"{len(train_rows)} images/epoch | {total_steps} optimizer steps | resume epoch {start_epoch}", flush=True)

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        ordered = list(train_rows)
        Random(2026 + epoch).shuffle(ordered)
        running_loss = 0.0
        start = perf_counter()
        for offset in range(0, len(ordered), accumulation):
            group = ordered[offset:offset + accumulation]
            weight_sum = sum(class_weights[label] for _, label in group)
            optimizer.zero_grad(set_to_none=True)
            for path, label in group:
                inputs = encode(processor, path, label)
                loss = model(**inputs, use_cache=False).loss
                if not torch.isfinite(loss).item():
                    raise FloatingPointError(f"Non-finite loss: {path}")
                weight = class_weights[label] / weight_sum
                (loss * weight).backward()
                running_loss += loss.detach().item()
            torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True)
            progress = max(0, step + 1 - warmup) / max(1, total_steps - warmup)
            scale = min(1, (step + 1) / warmup) * (0.1 + 0.9 * (1 + cos(pi * progress)) / 2)
            optimizer.param_groups[0]["lr"] = 1e-4 * scale
            optimizer.step()
            step += 1
            if step % 50 == 0:
                print(f"Step {step}/{total_steps} | epoch {epoch} | loss {running_loss / (offset + len(group)):.4f}", flush=True)
        optimizer.zero_grad(set_to_none=True)
        correct, confusion = evaluate(model, processor, splits["validation"])
        val_counts = Counter(label for _, label in splits["validation"])
        recalls = {label: confusion.get((label, label), 0) / val_counts[label] for label in labels}
        score = sum(recalls.values()) / len(labels)
        if score > best_score:
            best_score, best_epoch = score, epoch
        directory = run_dir / f"epoch_{epoch:02d}"
        model.save_pretrained(directory)
        state_path = directory / "training.pt"
        temporary = state_path.with_suffix(".tmp")
        torch.save(dict(config=config, optimizer=optimizer.state_dict(), epoch=epoch,
                        step=step, best_score=best_score, best_epoch=best_epoch,
                        cpu_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state(device)), temporary)
        temporary.replace(state_path)
        print(f"Epoch {epoch} | loss {running_loss / len(ordered):.4f} | val {correct}/{len(splits['validation'])} | macro recall {score:.1%} | {(perf_counter()-start)/60:.1f} min", flush=True)
        print({label: round(value, 3) for label, value in recalls.items()})
    return run_dir / f"epoch_{best_epoch:02d}"

In [ ]:
try:
    best_adapter = train_archive()
    print(best_adapter)
finally:
    gc.collect()
    torch.cuda.empty_cache()

## Held-out Base and LoRA evaluation

In [ ]:
base, processor = load_archive_model()
model = PeftModel.from_pretrained(base, best_adapter).eval()
with model.disable_adapter():
    baseline = evaluate(model, processor, splits["test"])
adapted = evaluate(model, processor, splits["test"])
support = Counter(label for _, label in splits["test"])
for name, (correct, confusion) in (("Base", baseline), ("LoRA", adapted)):
    recalls = {label: confusion.get((label, label), 0) / support[label] for label in labels}
    print(f"{name}: {correct}/{len(splits['test'])} | macro recall {sum(recalls.values()) / len(labels):.1%}")
    print({label: (confusion.get((label, label), 0), support[label]) for label in labels})
    for label in labels:
        predicted_count = sum(count for (_, pred), count in confusion.items() if pred == label)
        true_positive = confusion.get((label, label), 0)
        precision = f"{true_positive / predicted_count:.1%}" if predicted_count else "N/A"
        print(f"{label}: precision={precision}, recall={recalls[label]:.1%}, test support={support[label]}")
    print(confusion)
del model, base
gc.collect()
torch.cuda.empty_cache()

## Held-out evaluation with REVIEW

Run the first three setup code cells if the kernel restarted, then run the two code cells below. Each image is scored using mean generated-token log probability, excluding special tokens. Validation selects the largest coverage at 97% accepted accuracy. Test uses the selected threshold unchanged. Invalid labels and nonfinite scores always go to REVIEW.

In [ ]:
from math import isfinite
from statistics import mean

@torch.inference_mode()
def score_for_review(model, processor, rows):
    scored = []
    special_ids = set(processor.tokenizer.all_special_ids)
    for index, (path, truth) in enumerate(rows, start=1):
        inputs = encode(processor, path)
        output = model.generate(
            **inputs, max_new_tokens=16, do_sample=False,
            return_dict_in_generate=True, output_scores=True,
        )
        tokens = output.sequences[0, inputs.input_ids.shape[1]:]
        raw = processor.batch_decode(tokens.unsqueeze(0), skip_special_tokens=True)[0]
        answer = raw.strip().lower().removesuffix(".")
        candidate = answer.split("(", 1)[0].strip() if answer.endswith(")") else answer
        prediction = candidate if candidate in labels else "INVALID"
        log_probs = [
            logits[0].float().log_softmax(-1)[token].item()
            for token, logits in zip(tokens.tolist(), output.scores, strict=True)
            if token not in special_ids
        ]
        scored.append((path, truth, prediction, mean(log_probs) if log_probs else float("-inf")))
        if index % 100 == 0 or index == len(rows):
            print(f"  {index}/{len(rows)}", flush=True)
    return scored

def review_eligible(row):
    return row[2] in labels and isfinite(row[3])

def choose_review_threshold(rows, target=0.97):
    eligible = sorted((row for row in rows if review_eligible(row)),
                      key=lambda row: row[3], reverse=True)
    correct = 0
    threshold = float("inf")
    for count, (_, truth, prediction, score) in enumerate(eligible, start=1):
        correct += truth == prediction
        if (count == len(eligible) or eligible[count][3] != score) and correct / count >= target:
            threshold = score
    return threshold

def summarize_review(name, rows, threshold, per_class=False):
    print(f"\n{name} ({len(rows)} images)")
    counts = {label: Counter() for label in labels}
    for row in rows:
        _, truth, prediction, score = row
        accepted = review_eligible(row) and score >= threshold
        counts[truth]["total"] += 1
        counts[truth]["wrong"] += prediction != truth
        counts[truth]["accepted"] += accepted
        counts[truth]["wrong_accepted"] += accepted and prediction != truth

    total = sum(counts.values(), Counter())
    accepted = total["accepted"]
    errors = total["wrong_accepted"]
    print(f"  Baseline accuracy: {len(rows)-total['wrong']}/{len(rows)}")
    print(f"  Coverage: {accepted}/{len(rows)} ({accepted/len(rows):.1%})")
    print(f"  Accepted accuracy: {accepted-errors}/{accepted} ({(accepted-errors)/accepted:.1%})"
          if accepted else "  Accepted accuracy: N/A (all sent to REVIEW)")
    print(f"  REVIEW: {len(rows)-accepted} | errors caught: {total['wrong']-errors} | errors accepted: {errors}")
    if per_class:
        flagged = {row["label"] for row in rare_rows
                   if row["very_few_samples"] == "1" or row["test_support_below_10"] == "1"}
        print("True class | N | REVIEW | baseline errors | caught | accepted errors")
        for label, c in counts.items():
            if c["total"]:
                print(f"{label}{' *' if label in flagged else ''} | {c['total']} | "
                      f"{c['total']-c['accepted']} | {c['wrong']} | "
                      f"{c['wrong']-c['wrong_accepted']} | {c['wrong_accepted']}")
        print("* flagged in frozen rare-class metadata")


In [ ]:
review_adapter = root / "runs/qwen_archive_23_lora_v1/epoch_04"
print(f"Using {review_adapter}")
base, processor = load_archive_model()
try:
    model = PeftModel.from_pretrained(base, review_adapter).eval()
    try:
        print("Validation scoring")
        review_validation = score_for_review(model, processor, splits["validation"])
        review_threshold = choose_review_threshold(review_validation)
        print(f"Validation-selected threshold: {review_threshold:.9g}")
        summarize_review("Validation", review_validation, review_threshold)

        print("Test scoring")
        review_test = score_for_review(model, processor, splits["test"])
        summarize_review("Full test split", review_test, review_threshold, per_class=True)
        fresh_paths = {path for path, _ in fresh_test}
        summarize_review("Previously unused test subset",
                         [row for row in review_test if row[0] in fresh_paths],
                         review_threshold, per_class=True)
    finally:
        del model
finally:
    del base
    gc.collect()
    torch.cuda.empty_cache()